# Oral Cancer Detection — Tabular Pipeline
**Dataset:** NDB-UFES (237 real patients, Federal University of Espírito Santo, Brazil)

**Task:** Binary classification — OSCC (cancer) vs Leukoplakia (non-cancer)

**Models:** 9 models ranging from Logistic Regression to CatBoost and TabNet

---
### How to use this notebook
1. Upload `ndb-ufes.csv` when prompted in Cell 2
2. Run each cell top to bottom using **Shift+Enter**
3. All plots will appear inline below each cell

## 1. Install Dependencies

In [ ]:
!pip install catboost pytorch-tabnet shap --quiet

## 2. Upload Dataset

In [ ]:
from google.colab import files
print('Please upload ndb-ufes.csv')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

## 3. Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, f1_score, confusion_matrix, roc_curve, auc,
    matthews_corrcoef, cohen_kappa_score, log_loss
)
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from pytorch_tabnet.tab_model import TabNetClassifier

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
N_FOLDS = 5

FEATURE_NAMES = [
    'Tobacco Use', 'Alcohol Consumption', 'Sun Exposure', 'Gender', 'Age Group',
    'Loc: Floor of Mouth', 'Loc: Gingiva', 'Loc: Lip', 'Loc: Palate', 'Loc: Tongue',
    'Skin: Brown', 'Skin: Not Informed', 'Skin: White'
]

COLORS = {
    'Logistic Regression': '#4E79A7', 'Random Forest': '#F28E2B',
    'SVM': '#E15759',                 'XGBoost': '#76B7B2',
    'LightGBM': '#59A14F',            'MLP': '#EDC948',
    'CatBoost': '#B07AA1',            'TabNet': '#FF9DA7',
    'Stacking Ensemble': '#9C755F'
}

print('All imports successful.')

## 4. Preprocessing

In [ ]:
df = pd.read_csv('ndb-ufes.csv')
print(f'Raw shape: {df.shape}')
print(f'\nRaw target distribution:')
print(df['TaskII'].value_counts())

# Build binary target
df['label'] = (df['TaskII'] == 'OSCC').astype(int)

# Drop leakage and non-feature columns
DROP_COLS = ['public_id', 'lesion_id', 'patient_id', 'path',
             'diagnosis', 'dysplasia_severity', 'larger_size',
             'TaskII', 'TaskIII', 'TaskIV']
df = df.drop(columns=DROP_COLS)

# Encode features
for col in ['tobacco_use', 'alcohol_consumption']:
    df[col] = df[col].map({'No': 0, 'Former': 1, 'Yes': 2, 'Not informed': np.nan})
df['sun_exposure'] = df['sun_exposure'].map({'No': 0, 'Yes': 1, 'Not informed': np.nan})
df['gender'] = (df['gender'] == 'M').astype(float)

# Impute missing with mode
for col in ['tobacco_use', 'alcohol_consumption', 'sun_exposure']:
    df[col] = df[col].fillna(df[col].mode()[0])

# One-hot encode
df = pd.get_dummies(df, columns=['localization', 'skin_color'], drop_first=True, dtype=int)

print(f'\nCleaned shape: {df.shape}')
print(f'\nTarget distribution:')
print(df['label'].value_counts().rename({1: 'OSCC (cancer)', 0: 'Leukoplakia (non-cancer)'}))
print(f'\nMissing values: {df.isnull().sum().sum()}')
print('\nFeatures:')
print([c for c in df.columns if c != 'label'])

In [ ]:
# Train/test split (80/20, stratified)
X = df.drop(columns=['label']).values.astype(np.float32)
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

# Scale (fit on train only)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype(np.float32)
X_test_s  = scaler.transform(X_test).astype(np.float32)

neg, pos = np.sum(y_train == 0), np.sum(y_train == 1)
print(f'Train: {X_train_s.shape} | Test: {X_test_s.shape}')
print(f'Train class balance: OSCC={pos}, Leukoplakia={neg}')

## 5. Model Definitions

In [ ]:
def get_models(neg, pos):
    spw = neg / pos
    stacking = StackingClassifier(
        estimators=[
            ('rf',  RandomForestClassifier(n_estimators=300, random_state=RANDOM_SEED, class_weight='balanced', n_jobs=-1)),
            ('xgb', XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, scale_pos_weight=spw,
                                  random_state=RANDOM_SEED, eval_metric='logloss', verbosity=0, n_jobs=-1)),
            ('svm', SVC(kernel='rbf', probability=True, random_state=RANDOM_SEED, class_weight='balanced')),
        ],
        final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, class_weight='balanced'),
        cv=3, n_jobs=-1
    )
    return {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, class_weight='balanced', n_jobs=-1),
        'Random Forest':       RandomForestClassifier(n_estimators=300, random_state=RANDOM_SEED, class_weight='balanced', n_jobs=-1),
        'SVM':                 SVC(kernel='rbf', probability=True, random_state=RANDOM_SEED, class_weight='balanced'),
        'XGBoost':             XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, scale_pos_weight=spw,
                                             random_state=RANDOM_SEED, eval_metric='logloss', verbosity=0, n_jobs=-1),
        'LightGBM':            LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                                              class_weight='balanced', random_state=RANDOM_SEED, verbose=-1, n_jobs=-1),
        'MLP':                 MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', max_iter=500,
                                             random_state=RANDOM_SEED, early_stopping=True, validation_fraction=0.15),
        'CatBoost':            CatBoostClassifier(iterations=300, learning_rate=0.05, depth=4,
                                                  auto_class_weights='Balanced', random_seed=RANDOM_SEED, verbose=0),
        'TabNet':              TabNetClassifier(n_d=8, n_a=8, n_steps=3, gamma=1.3, n_independent=2,
                                               n_shared=2, seed=RANDOM_SEED, verbose=0),
        'Stacking Ensemble':   stacking
    }

def fit_predict(name, model, X_tr, y_tr, X_val):
    if name == 'TabNet':
        model.fit(X_tr, y_tr, max_epochs=200, patience=20, batch_size=32, virtual_batch_size=16)
    else:
        model.fit(X_tr, y_tr)
    return model.predict(X_val), model.predict_proba(X_val)[:, 1]

def compute_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'AUC-ROC':     round(roc_auc_score(y_true, y_prob), 4),
        'F1':          round(f1_score(y_true, y_pred), 4),
        'Sensitivity': round(tp / (tp + fn), 4),
        'Specificity': round(tn / (tn + fp), 4),
        'MCC':         round(matthews_corrcoef(y_true, y_pred), 4),
        'Cohen Kappa': round(cohen_kappa_score(y_true, y_pred), 4),
        'Log Loss':    round(log_loss(y_true, y_prob), 4),
    }

print('Model definitions ready.')

## 6. 5-Fold Cross-Validation

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
metric_names = ['AUC-ROC', 'F1', 'Sensitivity', 'Specificity', 'MCC', 'Cohen Kappa', 'Log Loss']
models_dict = get_models(neg, pos)
fold_scores = {name: {m: [] for m in metric_names} for name in models_dict}

print(f'Running {N_FOLDS}-fold stratified cross-validation (n={len(y)})...\n')

for name in models_dict:
    print(f'  {name}...')
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        sc = StandardScaler()
        X_tr_s  = sc.fit_transform(X_tr).astype(np.float32)
        X_val_s = sc.transform(X_val).astype(np.float32)
        n_neg, n_pos = np.sum(y_tr==0), np.sum(y_tr==1)
        model = get_models(n_neg, n_pos)[name]
        y_pred, y_prob = fit_predict(name, model, X_tr_s, y_tr, X_val_s)
        for m, v in compute_metrics(y_val, y_pred, y_prob).items():
            fold_scores[name][m].append(v)

# Build summary
summary = {}
raw_means = {}
for name in models_dict:
    summary[name] = {}
    raw_means[name] = {}
    for m in metric_names:
        vals = fold_scores[name][m]
        summary[name][m] = f'{np.mean(vals):.4f} +/- {np.std(vals):.4f}'
        raw_means[name][m] = np.mean(vals)

df_cv = pd.DataFrame(summary).T
df_cv.index.name = 'Model'

print('\n=== 5-FOLD CV RESULTS ===')
print(df_cv.to_string())

best_cv = max(raw_means, key=lambda m: raw_means[m]['AUC-ROC'])
print(f'\nBest model by AUC-ROC: {best_cv}  ({raw_means[best_cv]["AUC-ROC"]:.4f})')

## 7. Paired T-Test (Top 3 Models)

In [ ]:
ranked = sorted(raw_means, key=lambda m: raw_means[m]['AUC-ROC'], reverse=True)
top3 = ranked[:3]
print(f'Top 3 by CV AUC-ROC: {top3}\n')
print('Paired t-test on 5-fold AUC-ROC scores:')
pairs = [(top3[0], top3[1]), (top3[0], top3[2]), (top3[1], top3[2])]
ttest_results = {}
for a, b in pairs:
    t, p = stats.ttest_rel(fold_scores[a]['AUC-ROC'], fold_scores[b]['AUC-ROC'])
    sig = 'YES' if p < 0.05 else 'NO'
    print(f'  {a} vs {b}: t={t:.3f}, p={p:.4f} -> significant={sig}')
    ttest_results[f'{a} vs {b}'] = round(p, 4)

## 8. Holdout Test Set Evaluation

In [ ]:
print('Training all models on holdout split...\n')
holdout_models = get_models(neg, pos)
holdout_results = {}
all_preds = {}
all_probs = {}

for name, model in holdout_models.items():
    print(f'  {name}...')
    y_pred, y_prob = fit_predict(name, model, X_train_s, y_train, X_test_s)
    holdout_results[name] = compute_metrics(y_test, y_pred, y_prob)
    all_preds[name] = y_pred
    all_probs[name] = y_prob

df_holdout = pd.DataFrame(holdout_results).T
df_holdout.index.name = 'Model'
print('\n=== HOLDOUT TEST SET RESULTS ===')
print(df_holdout.to_string())

## 9. McNemar's Test

In [ ]:
top3_holdout = df_holdout['AUC-ROC'].nlargest(3).index.tolist()
pairs = [(top3_holdout[0], top3_holdout[1]), (top3_holdout[0], top3_holdout[2]), (top3_holdout[1], top3_holdout[2])]
print(f'Top 3 by holdout AUC-ROC: {top3_holdout}\n')
print("McNemar's test on holdout predictions:")
for a, b in pairs:
    ca = (all_preds[a] == y_test)
    cb = (all_preds[b] == y_test)
    bv = np.sum(ca & ~cb)
    cv = np.sum(~ca & cb)
    result = mcnemar([[0, bv], [cv, 0]], exact=False, correction=True)
    sig = 'YES' if result.pvalue < 0.05 else 'NO'
    print(f'  {a} vs {b}: chi2={result.statistic:.3f}, p={result.pvalue:.4f} -> significant={sig}')

## 10. ROC Curves — All 9 Models

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
for name, y_prob in all_probs.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    lw = 2.5 if name == 'CatBoost' else 1.5
    ls = '-' if name == 'CatBoost' else '--' if name in ['Random Forest', 'Stacking Ensemble', 'TabNet'] else ':'
    ax.plot(fpr, tpr, color=COLORS[name], lw=lw, ls=ls, label=f'{name}  (AUC = {roc_auc:.4f})')

ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5,label='Random (AUC = 0.5000)')
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
ax.set_title('ROC Curves — All Models (NDB-UFES Holdout Test Set)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.set_facecolor('#FAFAFA')
plt.tight_layout()
plt.show()

## 11. Confusion Matrices — Top 3 Models

In [ ]:
top3 = df_holdout['AUC-ROC'].nlargest(3).index.tolist()
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
fig.suptitle('Confusion Matrices — Top 3 Models (NDB-UFES Holdout Test Set)', fontsize=13, fontweight='bold')
labels = ['Non-Cancer\n(Leukoplakia)', 'Cancer\n(OSCC)']

for ax, name in zip(axes, top3):
    cm = confusion_matrix(y_test, all_preds[name])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm, annot=False, cmap='Blues', ax=ax, xticklabels=labels, yticklabels=labels,
                linewidths=0.5, linecolor='white', cbar=False, vmin=0, vmax=cm.max())
    for i in range(2):
        for j in range(2):
            color = 'white' if cm[i,j] > cm.max()*0.6 else 'black'
            ax.text(j+0.5, i+0.38, str(cm[i,j]), ha='center', va='center',
                    fontsize=16, fontweight='bold', color=color)
            ax.text(j+0.5, i+0.62, f'({cm_norm[i,j]*100:.1f}%)', ha='center', va='center',
                    fontsize=10, color=color)
    ax.set_title(name, fontsize=11, fontweight='bold', pad=10)
    ax.set_xlabel('Predicted Label', fontsize=10)
    ax.set_ylabel('True Label', fontsize=10)

plt.tight_layout()
plt.show()

## 12. Feature Importance — CatBoost vs Random Forest

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Feature Importance — CatBoost vs Random Forest (NDB-UFES)', fontsize=13, fontweight='bold')

fi_models = {
    'CatBoost':     (CatBoostClassifier(iterations=300, learning_rate=0.05, depth=4,
                     auto_class_weights='Balanced', random_seed=RANDOM_SEED, verbose=0), '#B07AA1'),
    'Random Forest':(RandomForestClassifier(n_estimators=300, random_state=RANDOM_SEED,
                     class_weight='balanced', n_jobs=-1), '#F28E2B'),
}

for ax, (name, (model, color)) in zip(axes, fi_models.items()):
    model.fit(X_train_s, y_train)
    fi = pd.Series(model.feature_importances_, index=FEATURE_NAMES).sort_values()
    colors = [color if v >= fi.quantile(0.6) else '#CCCCCC' for v in fi]
    bars = ax.barh(fi.index, fi.values, color=colors, edgecolor='white', height=0.7)
    for bar, val in zip(bars, fi.values):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8.5)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Importance Score', fontsize=10)
    ax.set_xlim(0, fi.max() * 1.18)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.set_facecolor('#FAFAFA')

plt.tight_layout()
plt.show()

## 13. SHAP Analysis — CatBoost

In [ ]:
print('Running SHAP analysis on CatBoost (best model)...')
cat_model = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=4,
                               auto_class_weights='Balanced', random_seed=RANDOM_SEED, verbose=0)
cat_model.fit(X_train_s, y_train)

explainer = shap.TreeExplainer(cat_model)
shap_values = explainer.shap_values(X_test_s)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('SHAP Analysis — CatBoost (NDB-UFES)', fontsize=13, fontweight='bold')

# Mean |SHAP| bar plot
mean_shap = np.abs(sv).mean(axis=0)
fi_shap = pd.Series(mean_shap, index=FEATURE_NAMES).sort_values()
colors_bar = ['#B07AA1' if v >= fi_shap.quantile(0.6) else '#CCCCCC' for v in fi_shap]
axes[0].barh(fi_shap.index, fi_shap.values, color=colors_bar, edgecolor='white', height=0.7)
for i, (idx, val) in enumerate(fi_shap.items()):
    axes[0].text(val + 0.001, i, f'{val:.3f}', va='center', fontsize=8.5)
axes[0].set_title('Mean |SHAP Value| per Feature', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Mean |SHAP Value|', fontsize=10)
axes[0].set_xlim(0, fi_shap.max() * 1.2)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)
axes[0].set_facecolor('#FAFAFA')

# Beeswarm
feature_order = np.argsort(mean_shap)
sv_ord = sv[:, feature_order]
fn_ord = [FEATURE_NAMES[i] for i in feature_order]
xv_ord = X_test_s[:, feature_order]

for i, (fname, shap_col, feat_col) in enumerate(zip(fn_ord, sv_ord.T, xv_ord.T)):
    jitter = np.random.RandomState(i).uniform(-0.25, 0.25, size=len(shap_col))
    norm_feat = (feat_col - feat_col.min()) / (np.ptp(feat_col) + 1e-8)
    colors_dot = plt.cm.RdBu_r(norm_feat)
    axes[1].scatter(shap_col, np.full_like(shap_col, i) + jitter,
                    c=colors_dot, s=18, alpha=0.75, linewidths=0)

axes[1].axvline(0, color='black', lw=0.8, ls='--', alpha=0.5)
axes[1].set_yticks(range(len(fn_ord)))
axes[1].set_yticklabels(fn_ord, fontsize=9)
axes[1].set_xlabel('SHAP Value (impact on model output)', fontsize=10)
axes[1].set_title('SHAP Beeswarm — Feature Impact per Sample', fontsize=11, fontweight='bold')
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)
axes[1].set_facecolor('#FAFAFA')

sm = plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(0,1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes[1], shrink=0.6, pad=0.02)
cbar.set_label('Feature value\n(low to high)', fontsize=8)
cbar.set_ticks([0, 1]); cbar.set_ticklabels(['Low', 'High'])

plt.tight_layout()
plt.show()

## 14. Save Results
Downloads CSV results to your local machine.

In [ ]:
df_cv.to_csv('ndbufes_cv_results.csv')
df_holdout.to_csv('ndbufes_holdout_results.csv')

files.download('ndbufes_cv_results.csv')
files.download('ndbufes_holdout_results.csv')
print('Results saved and downloaded.')